In [20]:
import os
print("Jupyter está corriendo desde:")
print(os.getcwd())

Jupyter está corriendo desde:
c:\Users\LENOVO\Documents\Big Data\Almacenes de datos\Proyecto_CanastaBasica\Canasta_Basica\etl


In [21]:
from pathlib import Path

# Definimos la ruta de inicio (Home)
home = Path.cwd()
print(home)
# Buscamos de forma recursiva todos los archivos .sav
# rglob significa "recursive glob" (busca en carpetas y subcarpetas)
#archivos = list(home.rglob("*.sav"))

#for archivo in archivos:
#    print(archivo)

c:\Users\LENOVO\Documents\Big Data\Almacenes de datos\Proyecto_CanastaBasica\Canasta_Basica\etl


In [22]:
#%pip install pyreadstat
import pyreadstat
import pandas as pd
from pathlib import Path
import os

# No es necesario cambiar de directorio - el notebook ya está en etl/
# os.chdir(Path.cwd())

BASE = Path("../data/raw")

archivos = {
    "enaho_2010":     BASE / "enaho/ENAHO 2010.sav",
    "enaho_2011":     BASE / "enaho/ENAHO 2011.sav",
    "enaho_2012":     BASE / "enaho/ENAHO 2012.sav",
    "enaho_2015":     BASE / "enaho/ENAHO 2015.sav",
    "enaho_2018":     BASE / "enaho/ENAHO 2018.sav",
    "enaho_2020":     BASE / "enaho/ENAHO 2020.sav",
    "enaho_2021":     BASE / "enaho/ENAHO 2021.sav",
    "enaho_2022":     BASE / "enaho/BdBasePublica 2022.sav",
    "enaho_2023":     BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2023_BdBasePublica.sav",
    "enaho_2024":     BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2024_BdPublica.sav",
    "enaho_2025":     BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2025_BdPublica.sav",
    "enigh_gastos":   BASE / "enigh/GPES-ELAB-GEBD-ENIGH-2024A_BdPublicaGastos.sav",
    "enigh_hogares":  BASE / "enigh/GPES-ELAB-GEBD-ENIGH-2024A_BdPublicaHogares.sav",
    "enigh_personas": BASE / "enigh/GPES-ELAB-GEBD-ENIGH-2024A_BdPublicaPersonas.sav",
}

resumen = []

for nombre, ruta in archivos.items():
    try:
        # Paso 1: obtener nombre de la primera columna
        _, meta = pyreadstat.read_sav(str(ruta), row_limit=0) # _ -> Variable desechable, meta -> Metadatos estructura
        primera_col = meta.column_names[0]
        cols_total  = len(meta.column_names)

        # Paso 2: leer solo esa columna para contar filas reales
        df, _ = pyreadstat.read_sav(str(ruta), usecols=[primera_col])
        filas = len(df)

        resumen.append({
            "archivo":       nombre,
            "filas":         filas,
            "columnas":      cols_total,
            "primera_col":   primera_col,
        })
        print(f"✅ {nombre:<20}: {filas:>8,} filas × {cols_total:>4} cols  | 1ra col: {primera_col}")

    except Exception as e:
        print(f"❌ {nombre}: {e}")

df_resumen = pd.DataFrame(resumen)
total = df_resumen["filas"].sum()
print(f"\n{'─'*55}")
print(f"  TOTAL FILAS brutas: {total:>12,}")

✅ enaho_2010          :   41,184 filas ×  493 cols  | 1ra col: FACTOR
✅ enaho_2011          :   40,860 filas ×  504 cols  | 1ra col: FACTOR
✅ enaho_2012          :   39,390 filas ×  517 cols  | 1ra col: FACTOR
✅ enaho_2015          :   37,291 filas ×  557 cols  | 1ra col: FACTOR
✅ enaho_2018          :   35,096 filas ×  565 cols  | 1ra col: FACTOR
✅ enaho_2020          :   25,530 filas ×  619 cols  | 1ra col: FACTOR
✅ enaho_2021          :   31,963 filas ×  595 cols  | 1ra col: FACTOR
✅ enaho_2022          :   31,045 filas ×  669 cols  | 1ra col: FACTOR
✅ enaho_2023          :   30,540 filas ×  594 cols  | 1ra col: FACTOR
✅ enaho_2024          :   30,846 filas ×  612 cols  | 1ra col: FACTOR
✅ enaho_2025          :   30,098 filas ×  596 cols  | 1ra col: FACTOR
✅ enigh_gastos        :  177,972 filas ×   21 cols  | 1ra col: ID_UPM
✅ enigh_hogares       :    6,107 filas ×  295 cols  | 1ra col: ID_UPM
✅ enigh_personas      :   17,006 filas ×  162 cols  | 1ra col: ID_UPM

───────────────────

In [10]:
# Ver columnas clave de enaho_2025 y enigh_gastos
for nombre, ruta in [
    ("enaho_2025",    BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2025_BdPublica.sav"),
    ("enigh_gastos",  BASE / "enigh/GPES-ELAB-GEBD-ENIGH-2024A_BdPublicaGastos.sav"),
]:
    _, meta = pyreadstat.read_sav(str(ruta), row_limit=0)
    print(f"\n{'='*55}")
    print(f"ARCHIVO: {nombre} — {len(meta.column_names)} columnas")
    print(f"\nNombres de columnas:")
    for i, (col, label) in enumerate(zip(meta.column_names, meta.column_labels)):
        print(f"  {i:>3}. {col:<25} | {label}")
    if i > 50:
        print(f"  ... y {len(meta.column_names)-50} columnas más")


ARCHIVO: enaho_2025 — 596 columnas

Nombres de columnas:
    0. FACTOR                    | Factor de expansión
    1. ID_HOGAR                  | Identifica el nivel hogar
    2. ID_VIVIENDA               | Identifica el nivel vivienda
    3. nro_Vivienda              | Número de vivienda anonimizado
    4. nro_Hogar                 | None
    5. HOGAR                     | Número del hogar
    6. LINEA                     | Número de línea
    7. REGION                    | Región de planificación
    8. ZONA                      | Zona
    9. V1                        | Tipo de vivienda
   10. V2A                       | Tipo de tenencia de la vivienda
   11. V2A1                      | Mensualidad pagada por la cuota del préstamo o el alquiler de la vivienda
   12. V2B                       | Mensualidad que pagaría si tuviera que alquilar
   13. V3                        | Principal material de las paredes exteriores de la vivienda
   14. V4                        | Principal mat

In [19]:
# Muestra de datos reales — columnas clave
import pandas as pd
import pyreadstat
from pathlib import Path

#BASE = Path("data/raw")

# ── Enaho 2025: solo columnas clave ──────────────────────
cols_enaho = [
    "FACTOR", "ID_HOGAR", "REGION", "ZONA",
    "TamHog",                          # tamaño del hogar
    "A4", "A5",                        # sexo, edad
    "NivInst",                         # nivel instrucción
    "CondAct",                         # condición actividad (ocupado/desocupado)
    "ipcb", "ipcn",                    # ingreso per cápita bruto/neto
    "ithb", "ithn",                    # ingreso total hogar bruto/neto
    "cba",                             # valor canasta básica alimentaria
    "lp",                              # línea de pobreza
    "np",                              # nivel de pobreza
    "Q_IPCN", "D_IPCN",               # quintil y decil
    "IPM_Pobreza",                     # pobreza multidimensional
]

df_enaho, _ = pyreadstat.read_sav(
    str(BASE / "enaho/GPES-ELAB-GEBD-ENAHO-2025_BdPublica.sav"),
    usecols=cols_enaho
)

print(f"Enaho 2025: {len(df_enaho):,} filas × {len(df_enaho.columns)} cols seleccionadas")
print(df_enaho.head(3).to_string())
print(f"\nNulos por columna:")
print(df_enaho.isnull().sum())
print(f"\nEstadísticas de ingreso y canasta:")
print(df_enaho[["ipcb","ithb","cba","lp"]].describe().round(0))

Enaho 2025: 30,098 filas × 19 cols seleccionadas
   FACTOR  ID_HOGAR  REGION  ZONA  TamHog   A4    A5  NivInst  CondAct       ithb       ithn      ipcb      ipcn      cba        lp   np  IPM_Pobreza  Q_IPCN  D_IPCN
0   253.0       1.0     1.0   1.0     3.0  2.0  39.0      3.0      1.0  1378000.0  1378000.0  459333.0  459333.0  61189.0  127150.0  3.0          0.0     4.0     7.0
1   253.0       NaN     1.0   1.0     3.0  1.0  21.0      7.0      3.0  1378000.0  1378000.0  459333.0  459333.0  61189.0  127150.0  3.0          0.0     4.0     7.0
2   253.0       NaN     1.0   1.0     3.0  2.0  15.0      3.0      3.0  1378000.0  1378000.0  459333.0  459333.0  61189.0  127150.0  3.0          0.0     4.0     7.0

Nulos por columna:
FACTOR             0
ID_HOGAR       19510
REGION             0
ZONA               0
TamHog             0
A4                 0
A5                 0
NivInst            0
CondAct         4932
ithb              38
ithn              38
ipcb              38
ipcn           

In [17]:
# ── ENIGH Gastos: muestra completa (solo 21 cols) ────────
df_gastos, _ = pyreadstat.read_sav(
    str(BASE / "enigh/GPES-ELAB-GEBD-ENIGH-2024A_BdPublicaGastos.sav")
)

print(f"\nENIGH Gastos: {len(df_gastos):,} filas × {len(df_gastos.columns)} cols")
print(df_gastos.head(5).to_string())
print(f"\nDivisiones de gasto únicas (CCIF 2018):")
print(df_gastos[["CCIF18_2DIG","DES_CCIF18_2DIG"]].drop_duplicates().sort_values("CCIF18_2DIG").to_string())
print(f"\nGasto mensualizado — estadísticas:")
print(df_gastos["GASTO_MENS"].describe().round(0))


ENIGH Gastos: 177,972 filas × 21 cols
  ID_UPM ID_VIVIENDA ID_HOGAR LLAVE_HOGAR ID_DECENA  ID_REGION  ID_ZONA CODIGO_CCIF18                               DESCRIPCION_COD_CCIF18  GASTO_MENS CCIF18_2DIG                                                   DES_CCIF18_2DIG CCIF18_3DIG                                DES_CCIF18_3DIG CCIF18_4DIG                                                    DES_CCIF18_4DIG  FACTOR  ESTRATO  QUINTIL_NACIONAL  DECIL_NACIONAL  QUINTIL_ZONA
0    504          03        1     1504031        35        5.0      1.0         05114          Mobiliario, alfombras y tapices sueltos (D)      2500.0          05  05.Mobiliario y equipo para el mantenimiento rutinario del hogar         051  051.Muebles, decoraciones y alfombras sueltas        0511                      0511.Muebles, enseres y alfombras sueltas (D)   185.0      4.0               1.0             1.0           1.0
1    524          03        1     1524031        31        5.0      2.0         05114          Mo